In [167]:
import pandas as pd
import importlib
import funciones as f #Tener funciones.py en mismo directorio. Tiene las funciones usadas para procesar un df
importlib.reload(f)

data = pd.read_csv('competition_data.csv')
submission = pd.read_csv('submission.csv')
submission_aux = pd.read_csv('submission.csv')

### Descomentar la siguiente celda la primera vez que se corre el notebook

In [168]:
# uri_to_ms_data = f.get_songs_durations(data)
# uri_to_ms_submission = f.get_songs_durations(submission)
# diccionario = {**uri_to_ms_data, **uri_to_ms_submission}

In [169]:
data['reason_start'].value_counts()

reason_start
fwdbtn        39765
trackdone     32827
clickrow      17796
playbtn        5548
backbtn        2572
appload         813
trackerror      422
remote          306
unknown          91
Name: count, dtype: int64

In [170]:
# Ejemplo de uso de procesar_df
submission = f.procesar_df(submission, diccionario)

In [171]:
submission

,Unnamed: 0,ts,platform,master_metadata_track_name,master_metadata_album_artist_name,master_metadata_album_album_name,spotify_track_uri,shuffle,hour,day_of_week,...,reason_fwdbtn,reason_playbtn,reason_remote,reason_trackdone,reason_trackerror,reason_unknown,track_prop,artist_prop,album_prop,hour day_of_week
0,74916,2014-06-27 18:01:15+00:00,"iOS 7.0.4 (iPod5,1)",Mejor,Los Tipitos,Push,spotify:track:5LFl6vXC2CwcciAbymL4jZ,False,18,4,...,False,False,False,False,False,False,0.000040,0.001837,0.000519,72.0
1,74923,2014-09-04 21:46:57+00:00,"iOS 7.0.4 (iPod5,1)","Circles - Based On Ludovico Einaudi ""Experience""",Ludovico Einaudi,In a Time Lapse,spotify:track:0mEsOEi4rWBy0IXE5oTKr2,False,21,3,...,True,False,False,False,False,False,0.000040,0.000240,0.000040,63.0
2,74924,2014-09-04 21:48:51+00:00,"iOS 7.0.4 (iPod5,1)",Primavera,Ludovico Einaudi,Divenire,spotify:track:0fzw4BBD5FRJtPuQbUUKzJ,False,21,3,...,False,False,False,False,False,False,0.000040,0.000240,0.000200,63.0
3,74933,2016-06-23 21:07:59+00:00,OS X 10.11.5 [x86 4],NaN,NaN,NaN,NaN,False,21,3,...,False,False,False,False,False,False,NaN,NaN,NaN,63.0
4,74934,2016-06-23 21:08:03+00:00,OS X 10.11.5 [x86 4],NaN,NaN,NaN,NaN,False,21,3,...,False,False,False,False,False,False,NaN,NaN,NaN,63.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25032,74898,2024-05-22 15:28:48+00:00,ios,Zafar,La Vela Puerca,A Contraluz,spotify:track:1wIUWGdTdhVk5gIPd0ULxX,True,15,2,...,False,False,False,True,False,False,0.000919,0.001598,0.001518,30.0
25033,74899,2024-05-22 15:35:04+00:00,ios,Un Loco En La Calesita,Juan Carlos Baglietto,Baglietto,spotify:track:3mHOEGxXbUpk5CZDgQhrUP,True,15,2,...,True,False,False,False,False,False,0.000639,0.002117,0.001558,30.0
25034,74900,2024-05-22 15:39:44+00:00,ios,Dulce condena - Edición Aniversario,Los Rodriguez,Sin Documentos,spotify:track:4Pk1N5mY14kO5N3JcADgb2,True,15,2,...,False,False,False,True,False,False,0.000719,0.006151,0.001238,30.0
25035,74901,2024-05-22 15:39:49+00:00,ios,Yo No Quiero Volverme Tan Loco,Charly García,Pubis Angelical / Yendo De La Cama Al Living,spotify:track:68LeIVjVDRMXPlfdFHhID6,True,15,2,...,False,False,False,True,False,False,0.001358,0.021288,0.002836,30.0


In [172]:
# Ordenar data cronológicamente
data = f.sort_by_ts(data)

In [173]:
# KFold
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
import numpy as np
import xgboost as xgb

def temporal_stratified_kfold(data, n_splits=5):
    '''
    Requiere: data esta ordenada cronologicamente y tiene una columna 'ts' con la fecha.
    Devuelve: los indices de entrenamiento y validación para cada fold en un KFold estratificado temporalmente.
    '''
    data['ts'] = pd.to_datetime(data['ts'])
    data['year'] = data['ts'].dt.year - 2000
    folds = []
    for fold in range(n_splits):
        idxs_train = []
        idxs_val = []
        for year, df_year in data.groupby('year'):
            df_year = df_year.sort_values('ts')

            fold_size = len(df_year) // n_splits
            val_start = fold * fold_size
            val_end   = (fold + 1) * fold_size if fold < n_splits - 1 else len(df_year)

            # .iloc aquí selecciona posiciones locales,
            # pero .index te devuelve los labels globales
            val_idx   = df_year.iloc[val_start:val_end].index
            train_idx = df_year.drop(val_idx).index

            idxs_train.extend(train_idx)
            idxs_val.extend(val_idx)
        folds.append((idxs_train, idxs_val))
    return folds

In [174]:
kf = temporal_stratified_kfold(data, n_splits=5)

auc_scores = []
columns_to_drop = ['Unnamed: 0', 'ts','TARGET', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'platform']

for fold, tupla in enumerate(kf):
    train_idx, val_idx = tupla
    print(f"Fold {fold + 1}")

    train_fold = data.loc[train_idx].copy().reset_index(drop=True)
    val_fold = data.loc[val_idx].copy().reset_index(drop=True)

    # Aplicar pipeline modular a cada fold
    train_fold = f.procesar_df(train_fold, diccionario)
    val_fold = f.procesar_df(val_fold, diccionario)
    print('len trainfold',len(train_fold.columns))
    print('len valfold',len(val_fold.columns))

    # Entrenar el modelo
    clf_xgb = xgb.XGBClassifier(objective = 'binary:logistic',
                            seed = 42,
                            eval_metric = 'auc',
                            early_stopping_rounds = 100)
    clf_xgb.fit(train_fold.drop(columns=columns_to_drop),
                train_fold['TARGET'],
                eval_set=[(val_fold.drop(columns=columns_to_drop), val_fold['TARGET'])],
                verbose=False
                )

    # Evaluar sobre validación
    preds = clf_xgb.predict_proba(val_fold.drop(columns=columns_to_drop))[:, 1]
    auc = roc_auc_score(val_fold['TARGET'], preds)
    auc_scores.append(auc)
    print(f"AUC Fold {fold + 1}: {auc:.4f}")

# Resultado final
print(f"\nAUC promedio: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")

Fold 1
len trainfold 29
len valfold 29
AUC Fold 1: 0.8652
Fold 2
len trainfold 29
len valfold 29
AUC Fold 2: 0.9179
Fold 3
len trainfold 29
len valfold 29
AUC Fold 3: 0.9082
Fold 4
len trainfold 29
len valfold 29
AUC Fold 4: 0.9122
Fold 5
len trainfold 29
len valfold 29
AUC Fold 5: 0.9274

AUC promedio: 0.9062 ± 0.0215


In [175]:
from sklearn.model_selection import TimeSeriesSplit

n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

# 3) Columnas a dropear / targets
drop_cols = [
    'Unnamed: 0','ts','TARGET',
    'master_metadata_track_name',
    'master_metadata_album_artist_name',
    'master_metadata_album_album_name',
    'spotify_track_uri','platform'
]

auc_scores = []
for fold, (train_idx, val_idx) in enumerate(tscv.split(data), start=1):
    print(f"Fold {fold}")

    train_fold = data.loc[train_idx].copy().reset_index(drop=True)
    val_fold   = data.loc[val_idx].copy().reset_index(drop=True)
    
    train_fold = f.procesar_df(train_fold, diccionario)
    val_fold   = f.procesar_df(val_fold,   diccionario)
    
    X_tr = train_fold.drop(columns=drop_cols)
    y_tr = train_fold['TARGET']
    X_va = val_fold  .drop(columns=drop_cols)
    y_va = val_fold  ['TARGET']
    
    clf = xgb.XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        seed=42,
        early_stopping_rounds=50
    )
    clf.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )

    preds = clf.predict_proba(X_va)[:, 1]
    auc   = roc_auc_score(y_va, preds)
    print(f"   AUC Fold {fold}: {auc:.4f}")
    auc_scores.append(auc)

# Resultados globales
print(f"\n✅ AUC mean: {np.mean(auc_scores):.4f} ± {np.std(auc_scores):.4f}")



🔁 Fold 1
   AUC Fold 1: 0.6434

🔁 Fold 2
   AUC Fold 2: 0.8412

🔁 Fold 3
   AUC Fold 3: 0.9166

🔁 Fold 4
   AUC Fold 4: 0.8524

🔁 Fold 5
   AUC Fold 5: 0.8182

✅ AUC mean: 0.8144 ± 0.0915
